# 🧠 `eval()` в Python: від «зручної функції» до «дірки в безпеці»

> **Урок:** Модулі, CLI, калькулятор  
> **Файл проекту:** `calculator_project/cli_calculator.py`

---

## 📋 План уроку

| # | Тема | Що дізнаємось |
|---|------|---------------|
| 1 | Що таке `eval()` | Синтаксис, базові приклади |
| 2 | `globals` та `locals` | Контроль середовища виконання |
| 3 | ⚠️ Небезпека `eval()` | Чому не можна довіряти вводу |
| 4 | Безпечні альтернативи | `ast.literal_eval`, словники |
| 5 | Алгоритм Shunting-Yard | Як парсити вирази без `eval()` |
| 6 | Наш калькулятор | Використання `cli_calculator.py` |
| 7 | Шпаргалка + Задачі | Підсумок та практика |

---

## 📦 Частина 1: Що таке `eval()`?

### Визначення

`eval()` — вбудована функція Python, яка **виконує рядок як Python-вираз** і повертає результат.

```
eval(expression, globals=None, locals=None)
```

| Параметр | Тип | Опис |
|----------|-----|------|
| `expression` | `str` | Рядок з Python-виразом |
| `globals` | `dict` або `None` | Глобальні змінні доступні у виразі |
| `locals` | `dict` або `None` | Локальні змінні доступні у виразі |

### Ключова ідея

```
┌─────────────────────────────────────────────────────────┐
│  eval("2 + 3 * 4")                                      │
│                                                         │
│   рядок "2 + 3 * 4"  →  Python парсить  →  виконує     │
│                                          →  повертає 14 │
└─────────────────────────────────────────────────────────┘
```

### Порівняння з `exec()`

| Функція | Що робить | Повертає |
|---------|-----------|----------|
| `eval(expr)` | Виконує **вираз** (expression) | Значення виразу |
| `exec(code)` | Виконує **код** (statements) | `None` |

In [ ]:
# Приклад 1.1: Базовий eval()
print("=== Базовий eval() ===")
print()

# eval() обчислює рядок як вираз
result = eval("2 + 3 * 4")
print(f"eval('2 + 3 * 4') = {result}")
print(f"Тип результату: {type(result).__name__}")
print()

# Різні типи виразів
expressions = [
    "10 + 5",
    "10 - 5",
    "10 * 5",
    "10 / 3",
    "10 // 3",
    "10 % 3",
    "2 ** 8",
    "(3 + 5) * 4",
]

print(f"{'Вираз':<20} {'Результат':<15} {'Тип'}")
print("-" * 50)
for expr in expressions:
    result = eval(expr)

    if isinstance(result, float):
        result = round(result, 3)

    print(f"{expr!r:<20} {result!r:<15} {type(result).__name__}")

In [ ]:
# Приклад 1.2: eval() бачить поточні змінні
print("=== eval() і змінні ===")
print()

number = 9
greeting = "Привіт"
pi = 3.14159

# eval() має доступ до всіх локальних змінних!
print(f"number = {number}")
print(f"eval('number * number')       = {eval('number * number')}")
print(f"eval('number ** 0.5')         = {eval('number ** 0.5'):.4f}")
print(f"eval('pi * 2')                = {eval('pi * 2'):.4f}")
print(f"eval('len(greeting)')         = {eval('len(greeting)')}")
print(f"eval('greeting.upper()')      = {eval('greeting.upper()')}")
print()
print("⚠️ eval() бачить ВСІ змінні поточного scope — це і зручно, і небезпечно!")

In [ ]:
# Приклад 1.3: eval() з функціями та модулями
import math

print("=== eval() з функціями ===")
print()

# Можна викликати функції через eval()
examples = [
    "math.sqrt(16)",
    "math.factorial(5)",
    "abs(-42)",
    "round(3.14159, 2)",
    "max(1, 5, 3, 2)",
    "[x**2 for x in range(5)]",  # навіть list comprehension!
]

for expr in examples:
    result = eval(expr)
    print(f"  eval({expr!r:<35}) → {result}")
print()
print("📌 eval() підтримує будь-який Python вираз — не тільки арифметику.")

---

## 🔧 Частина 2: Параметри `globals` та `locals`

`eval()` приймає два необов'язкові словники, що контролюють **яке середовище** бачить вираз.

```python
eval(expression, globals=None, locals=None)
```

### Як працює lookup у eval()

```
Пошук змінної у eval():
  1. locals  (якщо переданий)
  2. globals (якщо переданий)
  3. Стандартні builtins (якщо globals не забороняє)
```

### Таблиця сценаріїв

| Виклик | Що доступно у виразі |
|--------|----------------------|
| `eval(expr)` | Всі глобальні + локальні змінні + builtins |
| `eval(expr, {})` | Тільки builtins |
| `eval(expr, {'__builtins__': None})` | НІЧОГО (навіть `print` недоступний) |
| `eval(expr, {'sqrt': math.sqrt})` | Тільки `sqrt` + builtins |
| `eval(expr, {'__builtins__': None}, {'x': 5})` | Тільки `x` |

In [ ]:
# Приклад 2.1: globals — контроль доступних символів
import math

print("=== Контроль через globals ===")
print()

# Без обмежень — eval() бачить усе
result1 = eval("math.sqrt(16)")
print(f"1. Без обмежень:      eval('math.sqrt(16)') = {result1}")

# Передаємо порожній globals — math недоступний
try:
    result2 = eval("math.sqrt(16)", {})
except NameError as e:
    print(f"2. globals={{}}:         NameError: {e}")

# Передаємо тільки потрібні функції
allowed = {"sqrt": math.sqrt, "pi": math.pi}
result3 = eval("sqrt(16) + pi", allowed)
print(f"3. globals={{sqrt, pi}}: eval('sqrt(16) + pi') = {result3:.4f}")

print()
print("💡 globals дозволяє точно контролювати, які символи доступні у виразі.")

In [ ]:
# Приклад 2.2: Перейменування функцій через globals
import math

print("=== Перейменування функцій ===")
print()

# Дозволяємо функції під іншими іменами
custom_names = {
    "square_root": math.sqrt,
    "power":       pow,
    "cube":        lambda x: x ** 3,
    "__builtins__": None,  # забороняємо стандартні builtins
}

expressions = [
    "square_root(25)",
    "power(2, 10)",
    "cube(4)",
]

for expr in expressions:
    result = eval(expr, custom_names)
    print(f"  eval({expr!r:<25}) = {result}")
print()
print("💡 Можна надати користувачу 'безпечну мову' з власними іменами функцій.")

In [ ]:
# Приклад 2.3: globals + locals — два рівні видимості
import math

print("=== globals + locals ===")
print()

a = 169  # глобальна у нашому scope

# globals — базові функції, locals — конкретні змінні
globals_dict = {"__builtins__": None, "sqrt": math.sqrt}
locals_dict  = {"a": a, "b": 25}

result = eval("sqrt(a) + sqrt(b)", globals_dict, locals_dict)
print(f"a = {a}, b = {locals_dict['b']}")
print(f"eval('sqrt(a) + sqrt(b)', globals, locals) = {result}")
print()

# locals перекривають globals
globals_dict2 = {"x": 100}
locals_dict2  = {"x": 5}    # перекриває x=100
result2 = eval("x * 2", globals_dict2, locals_dict2)
print(f"globals x=100, locals x=5 → eval('x * 2') = {result2}  (locals перемогли)")

---

## ⚠️ Частина 3: Чому `eval()` небезпечний?

### Головне правило безпеки

> **Ніколи не передавайте в `eval()` дані від користувача без суворої фільтрації.**

### Що може зробити зловмисник через `eval(input())`?

```
┌─────────────────────────────────────────────────────────────┐
│  Ваш код:           print(eval(input("Вираз: ")))           │
│                                                             │
│  Звичайний ввід:    2 + 2               → 4  ✅             │
│                                                             │
│  Шкідливий ввід:                                            │
│    __import__("os").system("whoami")   → виконує команду ❌ │
│    __import__("os").system("ls /")     → читає файли  ❌    │
│    open("/etc/passwd").read()          → читає дані   ❌    │
│    __import__("shutil").rmtree(".")    → видаляє файли ❌   │
└─────────────────────────────────────────────────────────────┘
```

### Чому це можливо?

`eval()` виконує **будь-який Python вираз** — включно з:
- Викликом `__import__()` для завантаження будь-якого модуля
- Виконанням системних команд через `os.system()`
- Читанням/записом файлів
- Доступом до `__builtins__` і через них до всього Python

### Мем, який студенти запам'ятають

```
┌─────────────────────────────────────────────────────┐
│  Студент:                                           │
│  "Я зробив калькулятор через eval()!"               │
│                                                     │
│  Python-розробник:                                  │
│  "Ти зробив Remote Code Execution вразливість."     │
└─────────────────────────────────────────────────────┘
```

In [ ]:
# Приклад 3.1: Безпечна демонстрація небезпеки eval()
# (НЕ виконуємо шкідливий код — тільки аналізуємо)
print("=== Аналіз: що може статися з eval(input()) ===")
print()

# Симулюємо шкідливі введення (НЕ виконуємо!)
dangerous_inputs = [
    '2 + 2',
    '__import__("os").getcwd()',
    '__import__("os").system("whoami")',
    'open("passwords.txt").read()',
    '[x for x in ().__class__.__bases__[0].__subclasses__()]',
]

print(f"{'Ввід користувача':<45} {'Ризик'}")
print("-" * 70)
for inp in dangerous_inputs:
    if inp == '2 + 2':
        risk = "✅ безпечний"
    elif 'getcwd' in inp:
        risk = "⚠️  читає поточну папку"
    elif 'system' in inp:
        risk = "❌ ВИКОНУЄ системну команду"
    elif 'open' in inp:
        risk = "❌ ЧИТАЄ файли на диску"
    else:
        risk = "💀 дістає доступ до всіх класів Python"
    print(f"  {inp!r:<43} {risk}")
print()
print("💡 Висновок: eval(input()) — це фактично 'виконай будь-який код на моєму PC'.")

In [ ]:
# Приклад 3.2: Демонстрація — навіть 'безпечний' eval може бути зламаний
import os

print("=== Demo: навіть з обмеженнями eval() можна обійти ===")
print()

# Намагаємось захистити eval — забороняємо builtins
safe_globals = {"__builtins__": None}

# Звичайний вираз — працює
try:
    result = eval("2 + 2", safe_globals)
    print(f"1. eval('2 + 2', builtins=None) = {result}  ✅")
except Exception as e:
    print(f"1. Помилка: {e}")

# Намагаємось отримати доступ до os через ланцюжок класів
try:
    # Це класичний спосіб обходу eval sandbox
    # НЕ виконуємо шкідливий варіант — тільки перевіряємо чи він можливий
    exploit_expr = "().__class__.__bases__[0].__subclasses__()"
    subclasses = eval(exploit_expr, safe_globals)
    print(f"2. Через ланцюжок __class__ отримано {len(subclasses)} підкласів  ⚠️")
    print("   (зловмисник може знайти серед них os.popen або subprocess)")
except Exception as e:
    print(f"2. Заблоковано: {e}")

print()
print("🔑 Висновок: надійно захистити eval() від зловмисника — ДУЖЕ складно.")
print("   Найкращий захист — не використовувати eval() з user input взагалі.")

---

## 🛡️ Частина 4: Безпечні альтернативи `eval()`

### Варіант А: `ast.literal_eval()` — для даних

`ast.literal_eval()` безпечно парсить тільки **Python літерали**: числа, рядки, списки, словники, кортежі.

```python
import ast

ast.literal_eval("42")           # → 42
ast.literal_eval("[1, 2, 3]")    # → [1, 2, 3]
ast.literal_eval("{'key': 'val'}")  # → {'key': 'val'}
ast.literal_eval("2 + 2")        # → ValueError! (не літерал)
ast.literal_eval("os.system()")  # → ValueError! (не літерал)
```

### Варіант Б: Словник операторів — для калькулятора

```python
OPERATORS = {
    '+': lambda a, b: a + b,
    '-': lambda a, b: a - b,
    '*': lambda a, b: a * b,
    '/': lambda a, b: a / b if b != 0 else None,
}

def calculate(a, op, b):
    if op not in OPERATORS:
        raise ValueError(f"Невідомий оператор: {op}")
    return OPERATORS[op](a, b)
```

### Варіант В: Власний парсер (Shunting-Yard) — для складних виразів

Для повноцінного калькулятора з пріоритетами та дужками потрібен **алгоритм Shunting-Yard** — саме так реалізовано наш `cli_calculator.py`.

### Коли `eval()` прийнятний?

| Контекст | Безпечно? |
|----------|-----------|
| `eval("2 + 2")` — хардкодований рядок | ✅ Так |
| `eval(user_input)` — будь-який ввід | ❌ Ні |
| Скрипти для власного використання | ⚠️ Обережно |
| Веб-додатки, публічні API | ❌ Ніколи |

In [ ]:
# Приклад 4.1: ast.literal_eval() — безпечний парсинг даних
import ast

print("=== ast.literal_eval() ===")
print()

safe_inputs = [
    "42",
    "3.14",
    "[1, 2, 3]",
    "{'name': 'Alice', 'age': 25}",
    "(1, 2, 3)",
    "True",
]

unsafe_inputs = [
    "2 + 2",
    "os.system('ls')",
    "__import__('os')",
]

print("✅ Безпечні вхідні дані (літерали):")
for s in safe_inputs:
    try:
        result = ast.literal_eval(s)
        print(f"  ast.literal_eval({s!r:<30}) → {result!r}")
    except Exception as e:
        print(f"  ast.literal_eval({s!r:<30}) → Помилка: {e}")

print()
print("❌ Небезпечні вхідні дані (блокуються):")
for s in unsafe_inputs:
    try:
        result = ast.literal_eval(s)
        print(f"  ast.literal_eval({s!r:<30}) → {result!r}  ← ПРОЙШОВ (небезпека!)")
    except (ValueError, SyntaxError) as e:
        print(f"  ast.literal_eval({s!r:<30}) → ✅ ЗАБЛОКОВАНО: {type(e).__name__}")

In [ ]:
# Приклад 4.2: Словник операторів — простий безпечний калькулятор
print("=== Безпечний калькулятор: словник операторів ===")
print()

# Визначаємо дозволені операції
OPERATORS = {
    '+':  lambda a, b: a + b,
    '-':  lambda a, b: a - b,
    '*':  lambda a, b: a * b,
    '/':  lambda a, b: a / b if b != 0 else None,
    '//': lambda a, b: a // b if b != 0 else None,
    '%':  lambda a, b: a % b if b != 0 else None,
    '**': lambda a, b: a ** b,
}

def safe_calculate(a: float, op: str, b: float):
    """Безпечне обчислення без eval()."""
    if op not in OPERATORS:
        return None, f"Невідомий оператор: {op!r}"
    result = OPERATORS[op](a, b)
    if result is None:
        return None, "Ділення на нуль"
    return result, None

# Тестуємо
test_cases = [
    (10, '+', 5),
    (10, '-', 3),
    (4,  '*', 7),
    (15, '/', 4),
    (15, '//', 4),
    (15, '%', 4),
    (2,  '**', 8),
    (5,  '/', 0),
    (5,  '@', 3),  # невідомий оператор
]

print(f"{'Вираз':<20} {'Результат'}")
print("-" * 40)
for a, op, b in test_cases:
    result, err = safe_calculate(a, op, b)
    if err:
        print(f"  {a} {op} {b:<10} → ❌ {err}")
    else:
        print(f"  {a} {op} {b:<10} → {result}")

---

## ⚙️ Частина 5: Алгоритм Shunting-Yard

### Проблема простого словника операторів

Словник операторів добре працює для `a op b`, але не вміє:
- Враховувати **пріоритети**: `2 + 3 * 4 = 14`, не `20`
- Обробляти **дужки**: `(2 + 3) * 4 = 20`
- Обробляти **унарний мінус**: `-5 + 3 = -2`

### Рішення: алгоритм Shunting-Yard (Едсгер Дейкстра, 1961)

Алгоритм конвертує **інфіксний запис** (`2 + 3 * 4`) в **постфіксний (RPN)** (`2 3 4 * +`), де немає потреби в дужках і пріоритети вже враховані.

```
Інфіксний:   2 + 3 * 4
                  ↓  Shunting-Yard
Постфіксний: 2 3 4 * +   (RPN — Reverse Polish Notation)
                  ↓  обчислення стеком
Результат:   14
```

### Покрокове обчислення RPN

```
РПН: 2 3 4 * +

Стек:  []         читаємо 2    → стек: [2]
Стек:  [2]        читаємо 3    → стек: [2, 3]
Стек:  [2, 3]     читаємо 4    → стек: [2, 3, 4]
Стек:  [2, 3, 4]  читаємо *    → pop(4,3), push(3*4=12): [2, 12]
Стек:  [2, 12]    читаємо +    → pop(12,2), push(2+12=14): [14]
Результат: 14 ✅
```

### Три етапи в `cli_calculator.py`

```
  "2 + 3 * 4"
       ↓  tokenize()
  ["2", "+", "3", "*", "4"]
       ↓  to_rpn()      ← Shunting-Yard
  ["2", "3", "4", "*", "+"]
       ↓  eval_rpn()
  14.0
```

In [ ]:
# Приклад 5.1: Tokenizer — розбиття рядка на токени
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from cli_calculator import tokenize

print("=== Токенізація: рядок → список токенів ===")
print()

test_exprs = [
    "2 + 3 * 4",
    "(2 + 3) * 4",
    "-5 + 3.14",
    "2 ** 3 ** 2",
    "10 // 3 + 10 % 3",
    "2 @ 3",     # невідомий символ — помилка
]

for expr in test_exprs:
    ok, result = tokenize(expr)
    if ok:
        print(f"  {expr!r:<25} → {result}")
    else:
        print(f"  {expr!r:<25} → ❌ {result}")

In [ ]:
# Приклад 5.2: Shunting-Yard — інфікс → RPN
from cli_calculator import tokenize, to_rpn

print("=== Shunting-Yard: інфіксний → постфіксний (RPN) ===")
print()

test_exprs = [
    ("2 + 3 * 4",       "пріоритет * вище +"),
    ("(2 + 3) * 4",     "дужки змінюють пріоритет"),
    ("-5 + 3",          "унарний мінус"),
    ("2 ** 3 ** 2",     "** правоасоціативний"),
    ("1 + 2 + 3 + 4",   "ліва асоціативність"),
]

print(f"{'Вираз':<25} {'RPN':<30} {'Пояснення'}")
print("-" * 80)
for expr, note in test_exprs:
    ok1, tokens = tokenize(expr)
    ok2, rpn = to_rpn(tokens)
    if ok1 and ok2:
        print(f"  {expr!r:<23} {' '.join(rpn):<30} {note}")
    else:
        print(f"  {expr!r:<23} ❌ помилка")

In [ ]:
# Приклад 5.3: eval_rpn — обчислення постфіксного запису
from cli_calculator import tokenize, to_rpn, eval_rpn

print("=== eval_rpn: обчислення RPN ===")
print()

test_cases = [
    ("2 3 4 * +",       "2 + 3*4"),
    ("2 3 + 4 *",       "(2+3)*4"),
    ("5 u- 3 +",        "-5 + 3"),
    ("2 3 2 ** **",     "2**(3**2)"),
]

print(f"{'RPN-рядок':<25} {'Значить':<15} {'Результат'}")
print("-" * 55)
for rpn_str, meaning in test_cases:
    rpn_tokens = rpn_str.split()
    ok, result = eval_rpn(rpn_tokens)
    if ok:
        print(f"  {rpn_str!r:<23} {meaning:<15} = {result}")
    else:
        print(f"  {rpn_str!r:<23} ❌ {result}")

---

## 🖥️ Частина 6: Наш калькулятор `cli_calculator.py`

### Можливості

| Можливість | Деталі |
|------------|--------|
| Оператори | `+`, `-`, `*`, `/`, `//`, `%`, `**` |
| Дужки | `(`, `)` |
| Числа | цілі, дробові (3.14), унарний мінус (-5) |
| Команди CLI | `help`, `history`, `clear`, `exit` |
| sys.argv | `--help`, `--once "2+2"`, `--history N` |
| **eval()** | **Не використовується — власний парсер** |

### Запуск з командного рядка

```bash
# Інтерактивний режим
python cli_calculator.py

# Одноразове обчислення
python cli_calculator.py --once "2 ** 8"

# Довідка
python cli_calculator.py --help
```

In [ ]:
# Приклад 6.1: Повний pipeline — від рядка до результату
from cli_calculator import evaluate, format_number

print("=== Повний pipeline: рядок → результат ===")
print()

test_expressions = [
    "2 + 2",
    "10 - 3 * 2",
    "(10 - 3) * 2",
    "2 ** 8",
    "2 ** 3 ** 2",     # 2**(3**2) = 2**9 = 512 (права асоціативність)
    "-5 + 3",
    "3.14 * 2",
    "17 // 5",
    "17 % 5",
    "10 / 3",
    "10 / 0",          # ділення на нуль
    "abc + 1",         # некоректний вираз
    "(2 + 3",          # незакрита дужка
]

print(f"{'Вираз':<25} {'Результат'}")
print("-" * 50)
for expr in test_expressions:
    ok, result = evaluate(expr)
    if ok:
        formatted = format_number(result)
        print(f"  {expr:<25} = {formatted}")
    else:
        print(f"  {expr:<25} ❌ {result}")

In [ ]:
# Приклад 6.2: Порівняння eval() vs власний парсер
from cli_calculator import evaluate

print("=== eval() vs cli_calculator: порівняння ===")
print()

test_exprs = [
    "2 + 3 * 4",
    "(2 + 3) * 4",
    "2 ** 3 ** 2",
    "-5 + 10",
    "10 / 4",
    "10 // 4",
    "10 % 3",
]

print(f"{'Вираз':<25} {'eval()':<15} {'cli_calculator'}")
print("-" * 60)
for expr in test_exprs:
    eval_result = eval(expr)
    ok, calc_result = evaluate(expr)
    calc_str = str(calc_result) if ok else f"❌ {calc_result}"
    match = "✅" if ok and abs(eval(expr) - calc_result) < 1e-9 else "⚠️"
    print(f"  {expr!r:<23} {str(eval_result):<15} {calc_str} {match}")

print()
print("📌 Результати ідентичні — але cli_calculator безпечний для user input!")

In [ ]:
# Приклад 6.3: Безпека — cli_calculator відхиляє шкідливий ввід

from cli_calculator import evaluate

print("=== Безпека: cli_calculator відхиляє шкідливий ввід ===")
print()

attack_inputs = [
    '__import__("os").system("whoami")',
    'open("passwords.txt").read()',
    '[x for x in ().__class__.__bases__]',
    '"" * 10**9',   # DoS-спроба
    'print("hacked")',
]

print("eval() з цими виразами — НЕБЕЗПЕЧНО")
print("cli_calculator — БЕЗПЕЧНО відхиляє:\n")

for inp in attack_inputs:
    ok, result = evaluate(inp)
    text = repr(inp)[:50]

    if ok:
        print(f"  ⚠️ {text:<52} → {result} (пройшов!)")
    else:
        print(f"  ✅ {text:<52} → заблоковано: {result}")

---

## 📋 Шпаргалка (Quick Reference)

### eval() — синтаксис

```python
eval(expression)                           # доступ до всіх змінних
eval(expression, globals_dict)             # тільки вказані символи
eval(expression, globals_dict, locals_dict) # два рівні видимості
eval(expression, {'__builtins__': None})   # заборонити builtins
```

### Коли використовувати eval()

```python
# ✅ OK — хардкодований рядок
result = eval("2 + 2")

# ✅ OK — конфіг-файл, довіряємо джерелу
config = eval(trusted_config_string)

# ❌ НІКОЛИ — користувацький ввід
result = eval(input())          # вразливість!
result = eval(user_data)        # вразливість!
```

### Безпечні альтернативи

```python
# Для даних (числа, списки, словники):
import ast
value = ast.literal_eval(user_input)   # ValueError якщо не літерал

# Для простого a op b:
OPS = {'+': lambda a,b: a+b, '-': lambda a,b: a-b, ...}
result = OPS[op](a, b)

# Для складних виразів з пріоритетами:
from cli_calculator import evaluate
ok, result = evaluate(user_input)      # безпечний парсер
```

### Три етапи власного парсера

```python
# Рядок → токени → RPN → число
ok, tokens = tokenize("2 + 3 * 4")    # ['2', '+', '3', '*', '4']
ok, rpn    = to_rpn(tokens)            # ['2', '3', '4', '*', '+']
ok, result = eval_rpn(rpn)             # 14.0

# Або все одразу:
ok, result = evaluate("2 + 3 * 4")    # 14.0
```

---

# 📝 Задачі — Самостійна робота

### TODO 1: Безпечний eval з whitelist

Напишіть функцію `safe_eval(expr, allowed_functions)`, яка виконує вираз через `eval()`, але дозволяє тільки функції зі словника `allowed_functions`. Заблокуйте `__builtins__`.

```
allowed = {'sqrt': math.sqrt, 'abs': abs}
safe_eval('sqrt(16)', allowed)   # → 4.0
safe_eval('__import__("os")', allowed)  # → ValueError або None
```

In [ ]:
import math

def safe_eval(expr: str, allowed_functions: dict):
    """
    Виконує eval() тільки з дозволеними функціями.
    Забороняє __builtins__.
    Повертає результат або кидає виняток при порушенні.
    """
    # YOUR CODE HERE
    pass


# Тест (розкоментуйте після реалізації):
# allowed = {'sqrt': math.sqrt, 'pow': pow, 'abs': abs}
# print(safe_eval('sqrt(16)',         allowed))   # 4.0
# print(safe_eval('abs(-42)',         allowed))   # 42
# print(safe_eval('pow(2, 8)',        allowed))   # 256
# print(safe_eval('sqrt(16) + abs(-3)', allowed)) # 7.0
# try:
#     safe_eval('__import__("os")', allowed)
# except Exception as e:
#     print(f'Заблоковано: {e}')   # має кинути виняток

### TODO 2: Калькулятор без eval() — розширте функціонал

Додайте до функції `safe_calculate` підтримку унарного мінуса (коли `a = None`):

```
safe_calculate(None, 'u-', 5)  → -5
safe_calculate(None, 'u-', -3) → 3
```

In [ ]:
def safe_calculate(a, op: str, b: float):
    """
    Безпечне обчислення без eval().
    Якщо a is None — унарна операція (тільки op та b).
    """
    BINARY_OPS = {
        '+':  lambda a, b: a + b,
        '-':  lambda a, b: a - b,
        '*':  lambda a, b: a * b,
        '/':  lambda a, b: a / b if b != 0 else None,
        '//': lambda a, b: a // b if b != 0 else None,
        '%':  lambda a, b: a % b if b != 0 else None,
        '**': lambda a, b: a ** b,
    }
    UNARY_OPS = {
        # YOUR CODE HERE — додайте унарні оператори
    }

    # YOUR CODE HERE — реалізуйте логіку
    pass


# Тести:
# print(safe_calculate(10, '+', 5))     # 15
# print(safe_calculate(10, '/', 0))     # None (ділення на нуль)
# print(safe_calculate(None, 'u-', 5))  # -5
# print(safe_calculate(None, 'u-', -3)) # 3
# print(safe_calculate(None, 'u+', 7))  # 7

### TODO 3: Аналіз tokenizer

Подивіться на функцію `tokenize()` з `cli_calculator.py` та дайте відповіді:

1. Що поверне `tokenize("1.2.3")`? Поясніть чому.
2. Що поверне `tokenize("--5")`? Поясніть чому.
3. Що поверне `tokenize("3 ** -2")`? Поясніть чому.

In [ ]:
from cli_calculator import tokenize, evaluate

test_cases = [
    "1.2.3",
    "--5",
    "3 ** -2",
]

print("=== Аналіз tokenize() ===")
print()
for expr in test_cases:
    ok, result = tokenize(expr)
    print(f"tokenize({expr!r}):")
    print(f"  ok={ok}, result={result}")
    if ok:
        ok2, ev = evaluate(expr)
        print(f"  evaluate → {ev if ok2 else f'❌ {ev}'}")
    print()

# Ваші пояснення (замініть ... на свою відповідь):
# 1. "1.2.3" → ...
# 2. "--5"   → ...
# 3. "3**-2" → ...